# Gemma-Titans Hybrid: TPU v5e Online Learning Prototype

This notebook implements a hybrid architecture combining **Gemma-2B** with **Titans Neural Long-Term Memory (NLTM)**.

**Architecture Key Features:**
- **Parallel Memory:** Titans memory runs alongside standard Attention.
- **Learned Gating:** A scalar gate balances short-term context and long-term memory.
- **Modular Weights:** We only train and save the "Titans Delta" (~50MB), keeping the 5GB Gemma weights frozen.

## 1. Environment Setup (TPU v5e-1)
Run the cells below to install the JAX/Flax stack and the official Gemma framework.

In [3]:
#
!pip install tensorboard typeguard==4.4.1 gemma==3.3.0 flax==0.12.5 optax==0.2.6 einx kagglehub datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.2/225.2 kB 21.3 MB/s eta 0:00:00


In [1]:
# Clone the repository
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

/content/Titans_jax


## 2. Initialization & Model Surgery
We initialize the hybrid structure and extract the initial "Titans Delta" weights.

In [4]:
import jax
import jax.numpy as jnp
import os
import gc # Add garbage collection
# os.environ['KAULDRON_TYPECHECK'] = '0'
# os.environ['KD_CHECK_TYPES'] = '0'

# Nuclear option: Monkey-patch the internal kauldron type-checker
# try:
#     import kauldron.typing.type_check as kt_check
#     kt_check._check_argument_types = lambda *args, **kwargs: None
#     kt_check._check_return_type = lambda func, retval, *args, **kwargs: retval
#     print("Successfully monkey-patched kauldron type-checker.")
# except Exception as e:
#     print(f"Note: Could not monkey-patch kauldron: {e}")

from gemma.gm.nn import _config, _modules
from gemma_titans import GemmaTitansTransformer
from model_loader import stitch_hybrid_model, load_titans_delta

# 1. Define Gemma-2B Config
config_2b = _config.TransformerConfig(
    num_embed=256000,
    embed_dim=2048,
    hidden_dim=16384,
    num_heads=8,
    head_dim=256,
    num_kv_heads=1,
    final_logit_softcap=30.0,
    use_post_attn_norm=False,
    use_post_ffw_norm=False,
    attention_types=[_modules.AttentionType.GLOBAL] * 18,
)

model = GemmaTitansTransformer(config=config_2b, dtype=jnp.bfloat16)

print("Initializing Hybrid Structure (bfloat16)...")
rng = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((1, 1), dtype=jnp.int32)
# JIT compile init to run on TPU and prevent CPU RAM OOM
init_fn = jax.jit(model.init)
params = init_fn(rng, tokens=dummy_tokens)['params']
gc.collect() # Force free RAM
print("Initialization complete. CPU RAM freed.")

import orbax.checkpoint as ocp
import kagglehub
from gemma.gm import ckpts as gm_ckpts

# Extract only the Titans parameters (The Delta) from random init
def filter_titans(path, val):
    path_str = "/".join([str(p.name) if hasattr(p, 'name') else str(p) for p in path])
    if 'memory' in path_str or 'memory_gate' in path_str:
        return val
    return None

titans_delta = jax.tree_util.tree_map_with_path(filter_titans, params)

# Save initial random titans delta
print("Saving initial Titans Delta to ./titans_delta_init...")
checkpointer = ocp.StandardCheckpointer()
if os.path.exists("./titans_delta_init"):
    import shutil
    shutil.rmtree("./titans_delta_init")
checkpointer.save(os.path.abspath("./titans_delta_init"), titans_delta)

print("Downloading official Gemma-2B weights from Kaggle...")
# Note: Ensure you have authenticated with kagglehub or set KAGGLE_USERNAME and KAGGLE_KEY environment variables.
ckpt_path = kagglehub.model_download("google/gemma/Flax/2b")

print(f"Loading Gemma weights from {ckpt_path}...")
base_gemma_params = gm_ckpts.load_params(ckpt_path)

print("Stitching official Gemma weights with initialized Titans memory...")
params = stitch_hybrid_model(base_gemma_params, titans_delta)
print("Model loaded and stitched successfully!")
gc.collect()

Initializing Hybrid Structure (bfloat16)...
Initialization complete. CPU RAM freed.
Saving initial Titans Delta to ./titans_delta_init...



100%|██████████| 86.0/86.0 [00:00<00:00, 162kB/s]



100%|██████████| 29.7k/29.7k [00:00<00:00, 37.8MB/s]


  0%|          | 0.00/150 [00:00<?, ?B/s]

100%|██████████| 150/150 [00:00<00:00, 83.4kB/s]
100%|██████████| 56.0/56.0 [00:00<00:00, 75.2kB/s]


100%|██████████| 206/206 [00:00<00:00, 183kB/s]

...




100%|██████████| 55.0/55.0 [00:00<00:00, 137kB/s]


  2%|▏         | 17.0M/788M [00:00<00:14, 56.3MB/s]




100%|██████████| 85.0/85.0 [00:00<00:00, 213kB/s]





  0%|          | 0.00/4.04M [00:00<?, ?B/s]
  1%|          | 17.0M/2.89G [00:00<00:57, 54.0MB/s]

  3%|▎         | 24.0M/788M [00:00<00:17, 46.4MB/s]


 25%|██▍       | 1.00M/4.04M [00:00<00:00, 4.82MB/s]
  1%|          | 28.0M/2.89G [00:00<01:03, 48.4MB/s]


100%|██████████| 4.04M/4.04M [00:00<00:00, 12.6MB/s]


  4%|▍         | 30.0M/788M [00:00<00:20, 38.0MB/s]
  1%|▏         | 40.0M/2.89G [00:00<00:46, 66.5MB/s]
  2%|▏         | 50.0M/2.89G [00:00<00:40, 75.9MB/s]

  4%|▍         | 35.0M/788M [00:00<00:20, 37.9MB/s]
  2%|▏         | 62.0M/2.89G [00:00<00:35, 85.0MB/s]

  5%|▌         | 40.0M/788M [00:01<00:21, 35.7MB/s]
  2%|▏         | 72.0M/2.89G [00:01<00:33, 89.6MB/s]

  7%|▋         | 52.0M/788M [00:01<00:14, 52.5MB/s]
  3%|▎         | 82.0M/2.89G [00:01<00:32, 91.6MB/s]

  7%|▋         | 58.0M/788M [00:01<00:14, 51.8MB/s]
  3%|▎         | 93.0M/2.89G [00:01<00:30, 97.8MB/s]

  9%|▉         | 71.0M/788M [00:01<00:10, 71.6MB/s]
  4%|▎         | 105M/2.89G [00:01<00:29, 101M

Loading Gemma weights from /root/.cache/kagglehub/models/google/gemma/Flax/2b/2...


ValueError: No item metadata found in /root/.cache/kagglehub/models/google/gemma/Flax/2b/2

## 3. Dataset Preparation
We use a subset of OpenAssistant for dialogue training.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("OpenAssistant/oasst1", split="train[:100]")
print(f"Loaded {len(dataset)} dialogue samples.")

## 4. Phase 1: Continued Pre-Training (CPT)
We freeze the Gemma weights and train only the `memory` and `memory_gate` parameters.

In [ ]:
import optax

# Define the mask: Only train Titans parameters
def is_trainable(path, val):
    path_str = "/".join([str(p.name) for p in path])
    return 'memory' in path_str or 'memory_gate' in path_str

mask = jax.tree_util.tree_map_with_path(is_trainable, params)

# Setup Optimizer with masking
tx = optax.chain(
    optax.masked(optax.adam(learning_rate=1e-4), mask)
)
opt_state = tx.init(params)

@jax.jit
def train_step(params, opt_state, tokens):
    def loss_fn(p):
        logits = model.apply({'params': p}, tokens=tokens).logits
        # Simple Cross-Entropy Loss implementation would go here
        return jnp.mean(logits**2) # Dummy loss for prototype demo

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Starting training step...")
params, opt_state, loss = train_step(params, opt_state, dummy_tokens)
print(f"Initial loss: {loss}")

## 5. Dialogue Inference with Dynamic Memory
This demonstrates how the model updates its internal state (`TitansLayerCache`) during a conversation.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)
    logits = output.logits
    new_cache = output.cache

    # 2. Logic to extract the response token would go here
    return logits, new_cache

# Initial state
cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
user_tokens = jnp.array([[1, 2, 3, 4]])
logits, cache = dialogue_turn(params, cache, user_tokens)
print("Memory state updated automatically in cache.")

## 6. Save/Load Titans Delta
We save ONLY the trained memory weights to keep the file size minimal.

In [ ]:
import orbax.checkpoint as ocp

def save_titans_only(params, path):
    # Filter to save only Titans parameters
    titans_delta = jax.tree_util.tree_map_with_path(
        lambda path, val: val if is_trainable(path, val) else None,
        params
    )

    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(os.path.abspath(path), titans_delta, force=True)
    print(f"Saved Titans Delta to {path}")

save_titans_only(params, "./titans_delta_init")